# Probabilities with Rodalies data

<img src="https://www.totcerdanyola.cat/uploads/s1/27/24/55/65/62-estacio-de-cerdanyola-universitat-i-vies-de-la-linia-r7-de-rodalies-renfe.jpeg" alt="Cerdanyola Universitat station and the tracks of the R7 and R8 lines" width="700">

*Cerdanyola Universitat station, on our campus. Photo: Enric, from [Tot Cerdanyola](https://www.totcerdanyola.cat/actualitat/societat/mes-despres-descarrilament-uab-continua-sense-tren_2230734102.html) (21 February 2026).*

## Will your train be late?

A month after the **Gelida derailment** (January 2026), no Rodalies train reached the UAB: thousands of students had to swap the platform for a bus. Every day, the same question: *can I trust the train?*

"Trust" is a statement about **probabilities**. Today we learn their language and then apply it to **real Rodalies data** (46 days, from 9 October to 23 November 2026):

* How likely is a service to be delayed? Does it depend on the **line**, or on the **day**?
* Given a delayed service, which line is it likely to be?
* How long are the delays, and does a **normal distribution** describe them?

Every result in the notebook is labelled:

* **[OBSERVED]** counted directly from the data
* **[DERIVED]** computed from observed data (rates, means...)
* **[SIMULATED]** produced by a random simulation, *not* real operations
* **[MODEL]** what a mathematical model (e.g. the normal) predicts

## Topics

| # | Topic | Question it answers |
|---|---|---|
| 1 | Data | What are we looking at, and what can it tell us? |
| 2 | Basic probability | How likely is a delayed service? |
| 3 | Independence | Do delays on two lines happen independently? |
| 4 | Conditional probability | Does the delay depend on the line or the day? |
| 5 | Bayes' theorem | Given a delayed service, which line is it? |
| 6 | Monte Carlo simulation | What if the computer repeats the experiment? |
| 7 | PDF and CDF | How are the delays distributed? |
| 8 | Expectation and variance | What is the typical delay, and how variable is it? |
| 9 | Normal distribution | Is a normal curve a good model for delays? |
| 10 | Z-scores and inverse CDF | How unusual is a delay? Which delay is the 95th percentile? |
| 11 | Conclusion | What can (and cannot) the data tell us? |

# 1. Introduction and data

We use the public punctuality history published by [Trenes a tiempo](https://trenes.h1756.es/datos), derived from Renfe's real-time data (see the attribution at the end of the notebook). It has **one row per train number and service day**. The line that reaches the UAB is the **R8**: look out for it.

| Column | Meaning |
|---|---|
| `day` | Service day |
| `line` | Rodalies line of the train (added by us, see below) |
| `train_number` | Commercial train number |
| `observations` | Stops of that train with a reported delay |
| `instances` | Circulations of that train number that day with data (the denominator) |
| `on_time` | Circulations with reported delay $\le$ 2 minutes |
| `over_5min` | Circulations with reported delay $>$ 5 minutes |
| `delay_median_sec` | Median of the circulations' maximum reported delay that day (seconds) |
| `delay_p90_sec` | 90th percentile of those delays (seconds) |

**How we got the `line` column.** The original file has no line, and also includes Cercanías from the rest of Spain. We matched the train numbers with Renfe's official Cercanías timetable (GTFS) and kept only the **Barcelona nucleus (Rodalies)**. Trains whose number is also used in another nucleus are dropped, and a few train numbers run on more than one line (shown as `R11|R2N`). The script that did this is `build_rodalies_data.py`, next to this notebook.

In [ ]:
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

BASE = "https://raw.githubusercontent.com/cienciadadesdsuab/DS26-27/main/Laboratoris/Session%208%20-%20Probabilities/"

rod = pd.read_csv(BASE + "rodalies_puntualitat_2026-10-09_2026-11-23.csv", dtype={"train_number": str}, parse_dates=["day"])
print(rod.shape)
rod.head()

In [ ]:
print("[OBSERVED] first day:", rod["day"].min().date(), "| last day:", rod["day"].max().date(), "| days with data:", rod["day"].nunique())
print("[OBSERVED] train numbers:", rod["train_number"].nunique(), "| services with data:", rod["instances"].sum())
print()
print(rod.groupby("line")["instances"].sum().sort_values(ascending=False))

### Before using the numbers: what this data is (and is not)

* **Reported delay**, not verified arrival. It comes from Renfe's real-time systems.
* Each circulation counts with its **maximum** delay at *any* stop: a train that is late somewhere and recovers still counts as late.
* `on_time` ($\le$ 2 min) and `over_5min` ($>$ 5 min) **do not cover everything**: delays between 2 and 5 minutes are in neither (but they are in `instances`). So **never** compute on-time as $1 - P(\text{over 5 min})$.
* **Missing data is not punctuality.** Only days and trains with data appear, and some days have far fewer rows (for example 29 October, 10 and 11 November).
* A row is a **train number on a day**, not a passenger journey. Overall rates are always $\sum \text{delayed} / \sum \text{instances}$, never an average of row percentages.
* No **time of day**, so no rush hour. And no cancellations.
* **Not an official Renfe statistic.** A high share of delayed services here does not mean passengers experience that share of delays.

In [ ]:
# Helper columns
rod["weekday"] = rod["day"].dt.day_name()
rod["weekend"] = rod["day"].dt.dayofweek >= 5

# Trains that run on a single line (used to compare lines)
single_line = rod[~rod["line"].str.contains("|", regex=False)].copy()

# Individual circulations: when instances == 1 the daily "median" is the delay of that one circulation
circ = rod[rod["instances"] == 1].copy()
circ["delay_min"] = circ["delay_median_sec"] / 60

print("[OBSERVED] rows on a single line:", len(single_line), "of", len(rod))
print("[OBSERVED] rows that are exactly one circulation:", len(circ), "of", len(rod), f"({len(circ) / len(rod):.1%})")

The last cell matters for sections 7-10: when `instances == 1`, the median is not a summary, it is **the delay of that one circulation**, so we can study the real distribution of delays. We drop the few rows with two circulations instead of guessing.

# 2. Basic probability

It's a mathematical tool used to quantify the likelihood or chance of a specific event occurring.

We define the event

> **A**: a service has a reported delay greater than 5 minutes

and estimate its probability with the frequency of the event:

$$P(A) = \frac{\text{services with reported delay} > 5\ \text{min}}{\text{services with data}}$$

We **sum the counts first** and then divide.

In [ ]:
total_instances = rod["instances"].sum()
total_over_5 = rod["over_5min"].sum()
total_on_time = rod["on_time"].sum()

p_over_5 = total_over_5 / total_instances
p_on_time = total_on_time / total_instances

print("[OBSERVED] services with data:", total_instances)
print("[DERIVED]  P(reported delay > 5 min) =", round(p_over_5, 4))
print("[DERIVED]  P(reported delay <= 2 min) =", round(p_on_time, 4))
print("[DERIVED]  share in neither group (between 2 and 5 min):", round(1 - p_over_5 - p_on_time, 4))

In [ ]:
daily_all = rod.groupby("day")[["over_5min", "instances"]].sum()
daily_all["share"] = daily_all["over_5min"] / daily_all["instances"]

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(daily_all.index, daily_all["share"], marker="o")
ax.axhline(p_over_5, linestyle="--", color="gray", label="overall P(A)")
ax.set_ylabel("share of delayed services")
ax.set_title("Share of services with a reported delay > 5 min, day by day [DERIVED]")
ax.legend()
plt.show()

The last line is **not** the probability of being on time: it is the services whose delay is between 2 and 5 minutes. That is why `1 - P(A)` would be wrong as an on-time probability.

Also note that the share of delayed services is **high**. Remember limitation 2: a circulation counts with its *maximum* reported delay at *any* stop, and the delay is *reported* by real-time systems. This is not a passenger punctuality figure. Why could the number be so high?

**Your turn.** Compute the same probability for the weekend days only. Is it the same?

In [ ]:
# your code here

# 3. Independence

## Dependence and Independence  
2 events E and F are **dependent** if knowing something about whether E happens give us information about whether F happens ( and vice versa).
Otherwise they are **independent**.

2 events E and F are independent if the probability that they both happens is the product of the probabilities that each one happens:
                         **P(E,F) = P(E)P(F)**

### Let's see an example: (Independent)
In a family with 2 (unknown) children. If we assume that:


*   Each child is likely to be a boy or a girl
*   The gender of the second child is independent of the gender of the first child.

What is the probability of of the event of "no girls", the event of "one girl, one boy" and the event of " two girls"?

**Answer:**

When you have 2 children, the set of all possible outcomes (the sample space) is:

*   BB (Boy and Boy)
*   BG (Boy and Girl)
*   GB (Girl and Boy)
*   GG (Girl and Girl)



possible outcomes, and since the probability of having a boy or a girl is $\frac{1}{2}$ for each birth, each of these outcomes is equally likely (each has a probability of $\frac{1}{2} \times \frac{1}{2} = \frac{1}{4}$).

### Rodalies: are the delays of two lines independent?

We cannot pair arbitrary trains of two lines. Instead we compare the two lines **day by day**. For each line we call a day a *bad day* when its share of delayed services is above that line's median day:

* **A**: R1 has a bad day
* **B**: R2S has a bad day

If A and B were independent, we would expect $P(A, B) = P(A)\,P(B)$. By construction each event happens about half of the days, so independence would give a joint probability near 0.25.

In [ ]:
two = single_line[single_line["line"].isin(["R1", "R2S"])]
daily = two.groupby(["day", "line"])[["over_5min", "instances"]].sum()
daily["share_delayed"] = daily["over_5min"] / daily["instances"]
share = daily["share_delayed"].unstack()        # one row per day, one column per line

A = share["R1"] > share["R1"].median()
B = share["R2S"] > share["R2S"].median()

p_A, p_B, p_AB = A.mean(), B.mean(), (A & B).mean()
print("[DERIVED] days used:", len(share))
print("[DERIVED] P(A) =", round(p_A, 3), "| P(B) =", round(p_B, 3))
print("[DERIVED] P(A, B) =", round(p_AB, 3), "| P(A) * P(B) =", round(p_A * p_B, 3))
print("[DERIVED] correlation of the daily shares:", round(share["R1"].corr(share["R2S"]), 3))

In [ ]:
plt.scatter(share["R1"], share["R2S"])
plt.xlabel("R1: share of delayed services")
plt.ylabel("R2S: share of delayed services")
plt.title("One point per day [DERIVED]")
plt.show()

We have about six weeks of data, so these numbers are still noisy: a handful of days can easily move the joint probability. Still, delays on two lines can be linked when something affects the whole network (an incident in the shared infrastructure, bad weather, a strike, a holiday...). In this case a **common cause** (is it a weekday or a weekend?) can create dependence, as we explore in the next section.

# 4. Conditional probability

## Conditional Probability
In the case they are not necessarily independent, then we define the probability of E "conditional on F" as:
**P(E|F) = P(E,F)/P(F)**  
You can think of this as the probability that E happens, given that we know that F happens

### Let's see now another example (dependent)

What is the probability of the event "both children are girls" (b) conditional on the event " the older child is a girl" (G) ?

Using the definition of conditional probability:

**P(B|G) = P(B,G)/P(G) =$\frac{1}{2}$**

What is the probability of the event "both children are girls" (B) conditional on the event "at least one of the children is a girl (L)?
Surprisingly, the answer is different from before!

**P(B|L) = P(B,L)/P(L) =$\frac{1}{3}$**

Let's now make an experiment to validate this probabilities in a 10k trial

In [ ]:
import enum, random

# An Enum is a typed set of enumerated values. We can use them
# to make our code more descriptive and readable.
class Kid(enum.Enum):
    BOY = 0
    GIRL = 1

def random_kid() -> Kid:
    return random.choice([Kid.BOY, Kid.GIRL])

both_girls = 0
older_girl = 0
either_girl = 0

random.seed(0)

for _ in range(10000):
        younger = random_kid()
        older = random_kid()
        if older == Kid.GIRL:
            older_girl += 1 #This is the denominator for the first probability.
        if older == Kid.GIRL and younger == Kid.GIRL:
            both_girls += 1 # This is the numerator for both probabilities.
        if older == Kid.GIRL or younger == Kid.GIRL:
            either_girl += 1 #This is the denominator for the second probability.

print("P(both | older):", both_girls / older_girl)
print("P(both | either): ", both_girls / either_girl)

assert 0.48 < both_girls / older_girl < 0.52
assert 0.30 < both_girls / either_girl < 0.35

### Rodalies: delay probability given the line, or given the day type

$$P(A \mid L) = \frac{P(A, L)}{P(L)} = \frac{\text{delayed services on line } L}{\text{services on line } L}$$

We use the trains that run on a single line and only the lines with at least 500 services, so the groups are not too small.

In [ ]:
by_line = single_line.groupby("line").agg(services=("instances", "sum"), delayed=("over_5min", "sum"))
by_line["P(delayed | line)"] = by_line["delayed"] / by_line["services"]
by_line = by_line[by_line["services"] >= 500].sort_values("P(delayed | line)", ascending=False)
print("[DERIVED] conditional probability by line")
by_line.round(3)

In [ ]:
colors = ["tab:red" if line == "R8" else "tab:blue" for line in by_line.index]
by_line["P(delayed | line)"].plot.bar(color=colors, figsize=(8, 3.5))
plt.axhline(p_over_5, linestyle="--", color="gray")
plt.ylabel("P(delayed | line)")
plt.title("Delay probability by line, R8 (the UAB line) in red [DERIVED]")
plt.show()

In [ ]:
by_type = rod.groupby("weekend").agg(services=("instances", "sum"), delayed=("over_5min", "sum"))
by_type["P(delayed | day type)"] = by_type["delayed"] / by_type["services"]
by_type.index = ["weekday", "weekend"]
print("[DERIVED] conditional probability by day type")
by_type.round(3)

In [ ]:
by_day = rod.groupby("weekday").agg(services=("instances", "sum"), delayed=("over_5min", "sum"))
by_day["P(delayed | weekday)"] = by_day["delayed"] / by_day["services"]
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
print("[DERIVED] conditional probability by day of the week")
by_day.loc[order].round(3)

In [ ]:
by_day.loc[order, "P(delayed | weekday)"].plot.bar(figsize=(8, 3.5), color=["tab:blue"] * 5 + ["tab:orange"] * 2)
plt.axhline(p_over_5, linestyle="--", color="gray")
plt.ylabel("P(delayed | day)")
plt.title("Delay probability by day of the week [DERIVED]")
plt.show()

Two cautions when reading these tables:

* We have no time of day, so we cannot condition on rush hour.
* Monday 12 October is a national holiday in Spain and is counted here as a weekday. Check how many services ran that day compared with other Mondays.

### Our line: R8

The R8 (Martorell - Granollers, stopping at the UAB) is the line our campus depends on. Is it better or worse than the rest?

In [ ]:
is_r8 = single_line["line"] == "R8"
p_r8 = single_line.loc[is_r8, "over_5min"].sum() / single_line.loc[is_r8, "instances"].sum()
p_rest = single_line.loc[~is_r8, "over_5min"].sum() / single_line.loc[~is_r8, "instances"].sum()
print("[DERIVED] P(delayed | R8)          =", round(p_r8, 3))
print("[DERIVED] P(delayed | any other line) =", round(p_rest, 3))
print("[OBSERVED] R8 services in the data:", single_line.loc[is_r8, "instances"].sum())

**Your turn.** Is the difference big enough to say that the R8 is *really* different? Think about how many R8 services we have and how much the other lines vary among themselves.

# 5. Bayes' theorem

## Bayes's theorem
It's a way of "reversing" conditional probabilities. Let's say we need to know the probability of some event E conditional on some other event F occuring. But we only have information about the probability of F conditional on E occuring.
$$P(E|F) = \frac{P(E,F)}{P(F)} = \frac{P(F|E)P(E)}{P(F)}$$

Bayes' Theorem provides the mathematical bridge to make that reversal. But how did we get to that formula ?



1.   The probability of $E$ and $F$ is the probability of $F$ happening times the probability of $E$ happening given $F$:$$P(E,F) = P(E|F)P(F)$$

2.   The probability of $E$ and $F$ is the probability of $E$ happening times the probability of $F$ happening given $E$:$$P(E,F) = P(F|E)P(E)$$

3.   Since $P(E|F)P(F)$ and $P(F|E)P(E)$ are both equal to $P(E,F)$, they must be equal to each other:$$P(E|F)P(F) = P(F|E)P(E)$$

4.   Now, if you divide both sides by $P(F)$ (assuming $P(F) > 0$), you get the full formula for Bayes' Theorem:$$P(E|F) = \frac{P(F|E)P(E)}{P(F)}$$

5.   Replace P(T) using the law of total probability, either you have the disease or you don’t, so

$$P(F)=P(F∣E)P(E)+P(F∣Ec)P(Ec)$$

Combine with the previous expression to get Bayes’s theorem:
	​


$$P(E|F) = [ P(F|E)  P(E) ] / [ P(F|E)  P(E) + P(F|Ec) P(Ec) ]$$






## Let's see an example
 Imagine a certain disease that affects **1 in every 10,000 people**. And imagine that there is a test for this disease that gives the correct result ("diseased" if you have the disease , "nondiseased" if you don't)
**99%** of times.
What is the probability of a person that tested positive to actually have the disease ?
Let's use T for the event " your test is positive" and D for the event "you have the disease" .
Then Bayes's theorem says that the probability that you have the disease, conditional on testing positive is:


##  Replacement in Bayes's Theorem formula

Given:

- Disease prevalence  
  `P(E) = 1/10000 = 0.0001`

- True positive rate (sensitivity)  
  `P(F|E) = 0.99`

- False positive rate  
  `P(F|Ec) = 0.01`

- Probability of not having the disease  
  `P(Ec) = 1 - 0.0001 = 0.9999`


Bayes's theorem:

$$P(E|F) = [ P(F|E)  P(E) ] / [ P(F|E)  P(E) + P(F|Ec) P(Ec) ]$$

Substitute the known values:

**P(E)**     = 0.0001  (Probability of having the disease)

**P(F|E)**   = 0.99 (Probability of testing positive and having the disease)

**P(F|Ec)**  = 0.01 (Probability of testing positive and not having the disease)

**P(Ec)**    = 0.9999 (Probability of not having the disease)


So the formula becomes:

**P(E|F)** = (0.99 * 0.0001) / (0.99 * 0.0001 + 0.01 * 0.9999) ≈0.0098039216

Let's check it with a simulation. The disease is so rare that 10,000 people would contain only about one sick person, so we simulate **1,000,000**:

In [ ]:
import random

disease_count = 0
true_positive = 0
false_positive = 0

random.seed(0)

for _ in range(1_000_000):
    # whether the person has the disease
    has_disease = (random.random() < 1/10000)  # true only about 0.01% of the time

    if has_disease:
        disease_count += 1
        test_positive = (random.random() < 0.99)#true  about 99% of the time
    else:
        test_positive = (random.random() < 0.01) #true only about 1% of the time

    if test_positive:
        if has_disease:
            true_positive += 1
        else:
            false_positive += 1

P_D_given_T_sim = true_positive / (true_positive + false_positive) # Number of people who actually have the disease after testing positive

print("People with the disease:", disease_count)
print("P(D | T) simulated:", P_D_given_T_sim)
print("P(D | T) by Bayes:   ", 0.99 * 0.0001 / (0.99 * 0.0001 + 0.01 * 0.9999))


Even with a test that is right 99% of the time, a positive result means only about a **1% chance** of really having the disease: the disease is so rare that almost all positives are false alarms.

### Rodalies: given a delayed service, which line is it?

Instead of a disease, now the data is about trains. We know the rate of delays *within each line* and want the opposite: given that a service is delayed, the probability that it belongs to line R1:

$$P(R1 \mid \text{delayed}) = \frac{P(\text{delayed} \mid R1)\,P(R1)}{P(\text{delayed})}$$

All three quantities come from the same source and the same period (trains on a single line).

In [ ]:
n_total = single_line["instances"].sum()
n_R1 = single_line.loc[single_line["line"] == "R1", "instances"].sum()
n_delayed = single_line["over_5min"].sum()
n_delayed_R1 = single_line.loc[single_line["line"] == "R1", "over_5min"].sum()

P_R1 = n_R1 / n_total                           # share of services that are R1
P_delayed = n_delayed / n_total                 # share of services delayed
P_delayed_given_R1 = n_delayed_R1 / n_R1        # delay rate inside R1

bayes = P_delayed_given_R1 * P_R1 / P_delayed
direct = n_delayed_R1 / n_delayed               # count directly: delayed services that are R1

print("[DERIVED] P(R1) =", round(P_R1, 4))
print("[DERIVED] P(delayed) =", round(P_delayed, 4))
print("[DERIVED] P(delayed | R1) =", round(P_delayed_given_R1, 4))
print("[DERIVED] P(R1 | delayed) by Bayes  =", round(bayes, 4))
print("[DERIVED] P(R1 | delayed) by counts =", round(direct, 4))

Both ways give the same number, as they must: Bayes' theorem is just a rewriting of counts. Compare it with $P(R1)$: if line R1 has a higher share among the delayed services than among all services, R1 is over-represented among delays.

**Your turn.** Do the same for another line of your choice and for the weekend (given a delayed service, what is the probability that it ran on a weekend?).

In [ ]:
# your code here

# 6. Monte Carlo simulation

A **Monte Carlo simulation** repeats a random experiment many times and counts. We already used it for the children and the disease. Now we simulate 10,000 services in which each one is delayed with the probability we **observed**, $P(A)$.

> **[SIMULATED]** This is *not* a reconstruction of the real operation: the computer only knows the single number $P(A)$, so it cannot know which trains, lines or days were late.

In [ ]:
random.seed(0)
n_sim = 10_000

simulated_delayed = sum(random.random() < p_over_5 for _ in range(n_sim))

print("[OBSERVED] proportion delayed:", round(p_over_5, 4))
print("[SIMULATED] proportion delayed in", n_sim, "services:", round(simulated_delayed / n_sim, 4))

# How much does a simulation vary from run to run?
std_error = math.sqrt(p_over_5 * (1 - p_over_5) / n_sim)
print("[MODEL] expected standard error of the simulated proportion:", round(std_error, 4))

In [ ]:
random.seed(1)
runs = [sum(random.random() < p_over_5 for _ in range(n_sim)) / n_sim for _ in range(20)]
print("[SIMULATED] 20 runs, min / max:", round(min(runs), 4), "/", round(max(runs), 4))

The simulated proportion is close to the observed one but not identical, and it changes from run to run. With more trials the difference shrinks, as the standard error decreases like $1/\sqrt{n}$.

# 7. PDF and CDF

## PDF (Probability Density Function)
Function that describes the likelihood of a continuous random variable taking on a particular value.
Properties of a PDF:
*   𝑓𝑋(𝑥)≥0fX​(x)≥0  (probabilities cannot be negative).
*   The total area under the curve equals 1:
*   For continuous random variables, 𝑃(𝑋=𝑥)=0 for any exact value 𝑥
x; only intervals have non-zero probabilities.







In [ ]:
def uniform_pdf(x: float) -> float:
        return 1 if 0 <= x < 1 else 0 # Return 1 if X is between 0 and 1 for the uniform distribution

This function represents a distribution with equal likelihood for each event.

## CDF (Cumulative Distribution Function)
For a random variable **X** (continuous or discrete), the CDF of X, denoted by **𝐹𝑋(𝑥)**, gives the probability that X is less than or equal to a certain value x:
$$FX​(x)=P(X≤x)$$


In [ ]:
def uniform_cdf(x: float) -> float:
    """Returns the probability that a uniform random variable is <= x"""
    if x < 0:   return 0    # uniform random is never less than 0
    elif x < 1: return x    # e.g. P(X <= 0.4) = 0.4
    else:       return 1    # uniform random is always less than 1

### Rodalies: the distribution of the delay

From now on we use the **individual circulations** (`circ` table). The delay of a circulation is its maximum reported delay, in minutes. This is a *continuous* variable, so we describe it with a density (PDF) and a cumulative distribution (CDF), but now computed from the data: the **empirical** PDF is the histogram and the empirical CDF is the proportion of observations below each value.

In [ ]:
delay = circ["delay_min"]
print("[OBSERVED] circulations:", len(delay))
print(delay.describe().round(2))

In [ ]:
def empirical_cdf(values, x):
    """Proportion of observations less than or equal to x"""
    return (values <= x).mean()

print("[DERIVED] P(D <= 5 min)       =", round(empirical_cdf(delay, 5), 4))
print("[DERIVED] P(5 < D <= 10 min)  =", round(empirical_cdf(delay, 10) - empirical_cdf(delay, 5), 4))
print("[DERIVED] P(D > 10 min)       =", round(1 - empirical_cdf(delay, 10), 4))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(delay, bins=np.arange(0, 61, 1), density=True)
ax1.set_title("Empirical PDF (histogram, density) [OBSERVED]")
ax1.set_xlabel("Delay (minutes), shown up to 60")

xs = np.sort(delay.values)
ax2.plot(xs, np.arange(1, len(xs) + 1) / len(xs))
ax2.set_xlim(0, 120)
ax2.set_title("Empirical CDF [OBSERVED]")
ax2.set_xlabel("Delay (minutes), shown up to 120")
plt.show()

Look at the shape (and at the quartiles printed above): most circulations are concentrated at moderate delays, but there is a **long tail to the right** (a few very long delays). Keep that in mind for the normal distribution.

**Your turn.** What is the empirical probability that a circulation is delayed by more than 30 minutes?

In [ ]:
# your code here

# 8. Expectation and variance

For a sample $x_1,\dots,x_n$:

* **Mean (expectation)**: $\bar{x} = \frac{1}{n}\sum_i x_i$
* **Variance**: $s^2 = \frac{1}{n-1}\sum_i (x_i - \bar{x})^2$ and **standard deviation** $s = \sqrt{s^2}$ (same units as the delay)

We use the individual circulations, so every observation has the same weight. (With the daily summaries, an unweighted average of rows would be wrong, because rows represent different numbers of circulations.)

In [ ]:
n = len(delay)
mean_delay = sum(delay) / n
var_delay = sum((x - mean_delay) ** 2 for x in delay) / (n - 1)
std_delay = math.sqrt(var_delay)

print("[DERIVED] mean delay     :", round(mean_delay, 2), "min")
print("[DERIVED] variance       :", round(var_delay, 2), "min^2")
print("[DERIVED] std deviation  :", round(std_delay, 2), "min")
print("[DERIVED] median delay   :", round(delay.median(), 2), "min")
print("[CHECK]   numpy gives     ", round(np.mean(delay), 2), round(np.var(delay, ddof=1), 2))

In [ ]:
stats = circ[~circ["line"].str.contains("|", regex=False)].groupby("line")["delay_min"].agg(["count", "mean", "std", "median"])
stats = stats[stats["count"] >= 500].sort_values("mean", ascending=False)
print("[DERIVED] delay (minutes) by line")
stats.round(2)

In [ ]:
weekend_stats = circ.groupby("weekend")["delay_min"].agg(["count", "mean", "std", "median"])
weekend_stats.index = ["weekday", "weekend"]
print("[DERIVED] delay (minutes) by day type")
weekend_stats.round(2)

The mean is larger than the median: a few very long delays pull the mean up, which is another sign of a skewed distribution. Compare lines using both the mean and the median, and look at the standard deviation to see how variable each line is.

# 9. Normal distribution

## Normal Distribution

The Normal Distribution is the most important and widely used continuous probability distribution in statistics and probability theory.

It is a theoretical model that describes how the values of a continuous random variable are distributed in a population, with most of the data clustered around a central value, and the rest decreasing symmetrically as they move farther away from this center.

The distribution is completely defined by just two parameters:



1.   Mean ($\mu$): The central value (average), which defines the position of the curve's peak.
2.   Standard Deviation ($\sigma$): The measure of spread, which defines the width and height of the bell. A large $\sigma$ means a wider, flatter curve (greater dispersion); a small $\sigma$ means a narrower, more peaked curve (less dispersion).


####The Probability Density Function (PDF) of the Normal Distribution
Is the mathematical formula that describes the shape of the famous Bell Curve and determines the relative likelihood of a continuous random variable taking on any specific value.It's represented by the function $f(x)$ and is central to continuous probability distributions.

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}} e^{-\frac{1}{2}(\frac{x-\mu}{\sigma})^2}$$

where $$\frac{1}{\sigma\sqrt{2\pi}}$$  is normalizing constant.  
 $$\frac{x-\mu}{\sigma}^2$$  is shows how far $x$ is from the mean ($\mu$) in terms of standard deviations ($\sigma$). As $x$ moves away from $\mu$ (in either direction), the term becomes a large negative number, causing the value of $f(x)$ to rapidly approach zero, creating the "tails" of the bell curve.

In [ ]:
import math
SQRT_TWO_PI = math.sqrt(2 * math.pi)

def normal_pdf(x: float, mu: float = 0, sigma: float = 1) -> float:
    return (math.exp(-(x-mu) ** 2 / 2 / sigma ** 2) / (SQRT_TWO_PI * sigma))

import matplotlib.pyplot as plt
xs = [x / 10.0 for x in range(-50, 50)]
plt.plot(xs,[normal_pdf(x,sigma=1) for x in xs],'-',label='mu=0,sigma=1')
plt.plot(xs,[normal_pdf(x,sigma=2) for x in xs],'--',label='mu=0,sigma=2')
plt.plot(xs,[normal_pdf(x,sigma=0.5) for x in xs],':',label='mu=0,sigma=0.5')
plt.plot(xs,[normal_pdf(x,mu=-1)   for x in xs],'-.',label='mu=-1,sigma=1')
plt.legend()
plt.title("Various Normal pdfs")
plt.show()

#### The Cumulative Distribution Function in the Normal Function

It gives the total probability that a random variable $X$ will take on a value less than or equal to a specific value $x$.

The CDF is an accumulation of the probability density (the area under the PDF curve) starting from the far left (negative infinity) up to the point $x$.

The CDF is the integral of the Probability Density Function (PDF).
 $$F(x) = \int_{-\infty}^{x} f(t) dt$$

 Since the integral of the Normal PDF does not have a simple closed-form algebraic expression, the CDF values are calculated numerically (as you saw with the math.erf function) or looked up in Z-tables (for the Standard Normal Distribution).

 ** Use cases**


1.   Probability Less Than $x$ : $$P(X \le x) = F(x)$$
2.   Probability Greater Than $x$: $$P(X > x) = 1 - P(X \le x) = 1 - F(x)$$
3.   Probability Between Two Values: $$P(a < X \le b) = F(b) - F(a)$$

This is the function we will use to calculate cdf: $$P(X \le x) = \frac{1}{2} \left[1 + \text{erf}\left(\frac{x - \mu}{\sigma\sqrt{2}}\right)\right]$$
Where $(x - \mu) / \sigma$ is Z , transform a normal distribution into a standar normal distribution. $\text{math.erf}(z)$ is an special function to calculate the integral function of the gaussian funtion. And $(1 + \text{erf}(...)) / 2$ is a way to adjust the probability from -$-\infty$ to $x$.






In [ ]:
def normal_cdf(x: float, mu: float = 0, sigma: float = 1) -> float:
    return (1 + math.erf((x - mu) / math.sqrt(2) / sigma)) / 2

xs = [x / 10.0 for x in range(-50, 50)]
plt.plot(xs,[normal_cdf(x,sigma=1) for x in xs],'-',label='mu=0,sigma=1')
plt.plot(xs,[normal_cdf(x,sigma=2) for x in xs],'--',label='mu=0,sigma=2')
plt.plot(xs,[normal_cdf(x,sigma=0.5) for x in xs],':',label='mu=0,sigma=0.5')
plt.plot(xs,[normal_cdf(x,mu=-1) for x in xs],'-.',label='mu=-1,sigma=1')
plt.legend(loc=4) # bottom right
plt.title("Various Normal cdfs")
plt.show()

### Rodalies: does a normal distribution describe the delays?

The normal distribution is a **model that we must test**, not an assumption. We estimate $\mu$ and $\sigma$ from the delays and compare the model with the observed frequencies. Here the variable is the **non-negative lateness** of a circulation (its maximum reported delay, in minutes).

In [ ]:
mu, sigma = mean_delay, std_delay
print("[MODEL] normal with mu =", round(mu, 2), "and sigma =", round(sigma, 2))

observed = {
    "P(D <= 0)":      empirical_cdf(delay, 0),
    "P(D <= 5)":      empirical_cdf(delay, 5),
    "P(5 < D <= 10)": empirical_cdf(delay, 10) - empirical_cdf(delay, 5),
    "P(D > 30)":      1 - empirical_cdf(delay, 30),
}
model = {
    "P(D <= 0)":      normal_cdf(0, mu, sigma),
    "P(D <= 5)":      normal_cdf(5, mu, sigma),
    "P(5 < D <= 10)": normal_cdf(10, mu, sigma) - normal_cdf(5, mu, sigma),
    "P(D > 30)":      1 - normal_cdf(30, mu, sigma),
}
comparison = pd.DataFrame({"observed": observed, "normal model": model})
comparison.round(4)

In [ ]:
xs = [x / 2 for x in range(-60, 241)]     # -30 to 120 minutes
plt.hist(delay, bins=np.arange(-30, 121, 1), density=True, alpha=0.6, label="observed")
plt.plot(xs, [normal_pdf(x, mu, sigma) for x in xs], color="red", label="normal model")
plt.xlabel("Delay (minutes)")
plt.title("Observed delays vs normal model")
plt.legend()
plt.show()

skewness = sum((x - mean_delay) ** 3 for x in delay) / n / std_delay ** 3
print("[DERIVED] skewness:", round(skewness, 2), "(0 for a symmetric distribution)")

The comparison table and the picture answer the question: look at the probability of a **negative** delay predicted by the model, the probability of a very long delay, and the skewness. The normal fits poorly because the delays are not symmetric: they are almost never negative (hardly any train is reported early) and have a long right tail. For this kind of data a skewed model (for example the log-normal) would be a better starting point, but that is beyond today's lab.

# 10. Z-scores and inverse CDF

#### Z score

The Z-score (or Standard Score) is a statistical measure that tells you how many standard deviations ($\sigma$) a data point ($x$) is above or below the mean ($\mu$) of the distribution.

It's the  first step to standardize any Normal Distribution into the Standard Normal Distribution ($N(0, 1)$).  

$$Z = \frac{x - \mu}{\sigma}$$

Let's use Binary Search to find z:

In [ ]:
def inverse_normal_cdf(p: float,
                       mu: float = 0,
                       sigma: float = 1,
                       tolerance: float = 0.00001) -> float:
    """Find approximate inverse using binary search"""

    # if not standard, compute standard and rescale
    if mu != 0 or sigma != 1:
        return mu + sigma * inverse_normal_cdf(p, tolerance=tolerance)

    low_z = -10.0                      # normal_cdf(-10) is (very close to) 0
    hi_z  =  10.0                      # normal_cdf(10)  is (very close to) 1
    while hi_z - low_z > tolerance:
        mid_z = (low_z + hi_z) / 2     # Consider the midpoint
        mid_p = normal_cdf(mid_z)      # and the cdf's value there
        if mid_p < p:
            low_z = mid_z              # Midpoint too low, search above it
        else:
            hi_z = mid_z               # Midpoint too high, search below it

    return mid_z # Iterate until reaching the tolerance

For example, 95% of a standard normal lies below $z \approx 1.645$, and 97.5% below $z \approx 1.96$:

In [ ]:
print(inverse_normal_cdf(0.95))
print(inverse_normal_cdf(0.975))
print(normal_cdf(inverse_normal_cdf(0.975)))   # going back gives 0.975

### Rodalies: how unusual is a delay?

* The **Z-score** of a 20-minute delay tells us how many standard deviations it is from the mean.
* The **inverse CDF** gives the delay that corresponds to a percentile. We compute it with the normal model and compare with the **empirical** percentile.

In [ ]:
z_20 = (20 - mu) / sigma
print("[MODEL] Z-score of a 20-minute delay:", round(z_20, 2))
print("[DERIVED] share of circulations with a delay of 20 minutes or less:", round(empirical_cdf(delay, 20), 3))

In [ ]:
rows = []
for p in (0.90, 0.95):
    rows.append({"percentile": p,
                 "normal model (min)": inverse_normal_cdf(p, mu, sigma),
                 "empirical (min)": np.percentile(delay, p * 100)})
pd.DataFrame(rows).set_index("percentile").round(1)

If the normal model fitted well, both columns would be close. Compare them, and think about what each number would tell a passenger planning a trip. Also note that the Z-score of 20 minutes is a *mean-based* summary: when the data is skewed, the mean (and so Z) is a poor measure of what is unusual.

# 11. Conclusion

What did we learn?

* We can estimate probabilities from data as **frequencies**, always sum counts before dividing.
* **Conditional probabilities** let us compare groups (lines, weekdays) but we need consistent definitions and denominators, and enough data in every group.
* **Bayes' theorem** and plain counting give the same answer; Bayes shows how to reverse a conditional probability.
* **Simulations** reproduce the probability we put in. They do not discover anything the data does not contain.
* The **normal distribution** is a model, and a poor one for skewed, long-tailed delays.

And what the data **cannot** tell us: it contains *reported* delays (not verified arrivals), the *maximum* delay of each train, a limited period, no time of day, and no information on cancellations or on how passengers experienced the service. A high share of delayed services here is not the same as an unreliable service as seen by a passenger.

## Questions

**Question 1.** What is the observed probability of a reported delay greater than five minutes?

---

*Your Answer Here*

---

**Question 2.** Does it differ by line or between weekdays and weekends?

---

*Your Answer Here*

---

**Question 3.** Are the delay events of R1 and R2S independent? What could make them dependent?

---

*Your Answer Here*

---

**Question 4.** Given a delayed service, how does Bayes' theorem estimate its likely line?

---

*Your Answer Here*

---

**Question 5.** Does a normal distribution describe the observed delays well? Why (not)?

---

*Your Answer Here*

---

**Question 6.** How different are the empirical 95th percentile and the normal-model 95th percentile?

---

*Your Answer Here*

---

**Question 7.** What are the limitations of using real-time reported delay to describe the passenger experience?

---

*Your Answer Here*

---

---

### Data attribution

* *Datos de origen: Renfe Operadora (data.renfe.com), licencia CC BY 4.0. Derivados: Trenes a tiempo (trenes.h1756.es).* Period used: 9 October - 23 November 2026.
* Line assignment: Renfe Cercanías timetable (GTFS), Renfe Operadora, CC BY 4.0.